# RBD analysis — Step 1b: Build the protein FASTA

Take the model's predicted-positive interactions (`prediction == 1`) on the redundant-split fold 1596 and emit one FASTA record per unique protein. This FASTA is the input universe for `run_hmmscan.sh`.

**Inputs**
- `data/predictions/train_1596_predicted.csv`
- `data/predictions/val_1596_predicted.csv`

**Output**
- `data/sequences/predicted_interacting_proteins.fa`

Run from the repo root, after `01_build_rbd_whitelist.ipynb`.

In [6]:
import sys
import pandas as pd
from pathlib import Path

# Repo-root bootstrap: makes `coral` importable without installing the package.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").is_file() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
from coral import paths

In [7]:
train = pd.read_csv(paths.PREDICTIONS_DIR / "train_1596_predicted.csv")
val = pd.read_csv(paths.PREDICTIONS_DIR / "val_1596_predicted.csv")
all_pairs = pd.concat([train, val], axis=0)
all_interactions = all_pairs.loc[all_pairs.prediction == 1]
all_interactions

,RNA_id,Protein_id,PDB_id,RNA_seqs,Prot_seqs,prediction,interaction_probability,non_interaction_probability,true_label
0,ENST00000636325,P23246,P23246,TACGGATGAGCAAAGAAAGTGGTTTCTTAAAATGGAATCTACTCTT...,MSRDRFRSRGGGGGGFHRRGGGGGRGGLHDFRSPPPGMGLNQNRGP...,1,0.999912,0.000088,1
1,ENST00000458063,Q8TCT6,Q8TCT6,GGTGAACTGGACTTGGTGGATGGTGTGGAAGGGACTTTGGGAGAGA...,MAEQTYSWAYSLVDSSQVSTFLISILLIVYGSFRSLNMDFENQDKE...,1,0.999658,0.000343,1
2,ENST00000684525,Q8TDD1,Q8TDD1,AGACCAGGAAGCGGATAGCGTGGAGTGACGGTGCCACCGCGGCGCA...,MAADKGPAAGPRSRAAMAQWRKKKGLRKRRGAASQARGSDSEDGEF...,1,0.999156,0.000844,1
3,ENST00000504440,Q9HCK5,Q9HCK5,GAGGCGGTCCCCATGGCGTCCGTTGCCGCTACTACCTAGAATTGGT...,MEALGPGPPASLFQPPRRPGLGTVGKPIRLLANHFQVQIPKIDVYH...,1,0.999111,0.000889,1
4,ENSMUST00000143857,Q8BP71,Q8BP71,GTTCAAAATCAGATTTGGTGGATGAAACAAAAGATCCAGGCATGGG...,MAEGGQAQQQPPQLGPGAAARGMKRESEVELPVPGAGADGPEPGLS...,1,0.999879,0.000121,1
...,...,...,...,...,...,...,...,...,...
5034,ENST00000453716,Q12926,Q12926,CACGGTGTGGCCTCAGCATCAGGGATATTTGCAACATCCATATGGC...,METQLSNGPTCNNTANGPTTINNNCSSPVDSGNTEDSKTNLIVNYL...,1,0.995354,0.004646,1
5035,MIMAT0000123,Q61039,Q61039,TGGAATGTAAAGAAGTATGTAT,MSLVGGFPHHPVVHHEGYPFAAAAAAAAAAAASRCSHEENPYFHGW...,1,0.997177,0.002823,1
5038,YLR459W-MRNA,P38199,P38199,ATGGATTCCACAGCACTTAAGGTAGCTCTAGGCTGTATTGCAATTC...,MSQFFEAATPVAIPTNNTNGGSSDAGSAATGGAPVVGTTAQPTINH...,1,0.999749,0.000251,1
5039,MIMAT0000077,P09382,P09382,AAGCTGCCAGTTGAAGAACTGT,MACGLVASNLNLKPGECLRVRGEVAPDAKSFVLNLGKDSNNLCLHF...,1,0.830380,0.169620,1


In [ ]:
proteins = all_interactions.loc[:, ["Protein_id", "Prot_seqs"]].drop_duplicates()

with open(paths.SEQUENCES_DIR / "predicted_interacting_proteins.fa", "w") as f:
    for _, protein_id, prot_seq in proteins.itertuples():
        f.write(f">{protein_id}\n{prot_seq}\n")

In [ ]:
all_interactions = all_interactions.loc[:, ["RNA_id", "Protein_id", "RNA_seqs", "Prot_seqs", "prediction"]]
all_interactions.to_csv(paths.PREDICTIONS_DIR / "all_predicted_interactions.csv", index=False)